# 01. Tries

**Tier:** Deep. **Why you'd use this:** You store many words that share prefixes. A tree of letters answers "is this word present?" without scanning every stored string from the start.

**After this lesson you can:** insert and search words in a letter tree; mark word ends so a prefix is not mistaken for a word.
About 25 minutes. **Uses:** [Dicts](../../course/04-collections/03-dicts-lesson.ipynb), [Classes](../../course/11-classes/01-classes-lesson.ipynb)


## Quick review

1. What does each edge in a letter tree usually represent?
2. Why mark the end of a word instead of only storing letters?

<details><summary>Answers</summary>

One next character. Without an end mark, every prefix of a stored word would look like a hit.
</details>


## Recognize it

Many strings share prefixes, and you need repeated "is this word in the collection?" or "does any word start with this prefix?" checks.

Signals: a dictionary of words, autocomplete, or prefix queries where a flat list would rescan shared letters. A look-alike that is not this pattern: one-shot "longest common prefix of a fixed list" — a linear scan of the first word is enough, and a tree is optional.

If every query is a full string and you never ask about stems, a `set` is simpler. The trie earns its keep when the shared structure itself is part of the query workload.


## Why it works this way

A trie is a tree whose edges are characters. Each node holds a map from the next character to a child node, plus a boolean that means "a word ends here". Insert walks one edge per character, creating children as needed, and sets the mark on the final node. Search walks the same path and returns true only if every character exists and the final node is marked.

Shared prefixes share nodes. The words `hat` and `hi` both travel through `h`, then branch. A flat list of strings stores that `h` once per word and compares from the start on every lookup. Lookup in the tree costs time proportional to the query length, not to how many words you stored, when the branching stays in hash maps. See [Dicts](../../internals/04-dicts/01-hash-table-lesson.ipynb) for why each node's child map is average-constant per character.

Memory grows with the total number of distinct prefixes, which can be less than the total characters across words when overlap is high, or similar when words barely share. The end mark is the design detail that separates "path exists" from "word was inserted". Without it, `search("h")` would succeed after inserting only `"hi"`.

A Python `set` of whole words also answers membership in average-constant time per full string hash, and it is simpler when you never need prefix queries. The trie earns its keep when many queries are prefixes, when you walk all words under a stem, or when you build patterns character by character. Listing every stored word is a depth-first walk that emits a string whenever a mark is true.

Prefix search stops early on a missing edge and returns false without scanning unrelated words. That early exit is the practical win over a linear scan of a dictionary list: failed lookups pay only for the shared stem they managed to match, not for every stored string's full compare.


## When to use it

Use a letter tree when membership and prefix checks repeat over a growing dictionary, or when you must list words under a stem. Prefer a `set` of strings for whole-word membership with no prefix work. Prefer a single scan for a one-off longest common prefix of a small fixed list. Prefer a hash map from full string to value when you never care about shared structure.

For huge alphabets or sparse branching, child dicts stay flexible. Fixed 26-slot arrays are a micro-optimization for lowercase English only and are not required for correctness.


## Brute force, and why it's slow

Count character comparisons while searching a flat list for a word.


In [1]:
def search_slow(words, query):
    comparisons = 0
    for word in words:
        n = min(len(word), len(query))
        for i in range(n):
            comparisons += 1
            if word[i] != query[i]:
                break
        else:
            if len(word) == len(query):
                return True, comparisons
            comparisons += 1
    return False, comparisons


words = [f"pre{i}" for i in range(200)] + ["prefix"]
for size in (50, 100, 200):
    print(size, search_slow(words[:size] + ["prefix"], "prefix"))

50 (True, 206)
100 (True, 406)
200 (True, 806)


## Key insight

One node per character on a path, shared across words. Search follows the path and trusts the end mark, not merely a surviving path. Prefix queries stop at the stem without needing a mark.


## Worked example

Build a small word book. Insert words, then search for whole words. A proper prefix of an inserted word is not a hit unless you inserted that prefix on its own.


In [2]:
class Book:
    def __init__(self) -> None:
        # 1. Children keyed by next character; word marks an end.
        self.children: dict[str, Book] = {}
        self.word = False

    def add(self, text: str) -> None:
        node = self
        for character in text:
            node = node.children.setdefault(character, Book())
        # 2. Mark the last node as a finished word.
        node.word = True

    def has_word(self, text: str) -> bool:
        node = self
        for character in text:
            if character not in node.children:
                return False
            node = node.children[character]
        # 3. Path alone is not enough; the mark decides.
        return node.word


book = Book()
book.add("hi")
book.add("hat")
print(book.has_word("hi"), book.has_word("h"), book.has_word("hat"))

True False True


## Trace

Insert `"hi"`, then `"hat"`:

| step | path built | marks |
|---|---|---|
| add `hi` | root → `h` → `i` | `i.word = True` |
| add `hat` | root → `h` → `a` → `t` | `t.word = True` (`h` still unmarked) |

`has_word("h")` → `False`. `has_word("hi")` → `True`.


## Cost

Insert and search walk one node per character: time grows with the query length. Memory grows with distinct prefix nodes. That beats a flat scan whose comparisons grow with the number of stored words. Failed lookups exit at the first missing edge instead of finishing every dictionary compare.


## Variants

Not solved here: [Implement Trie](../../challenges/problems/39-implement-trie.ipynb), [Word Search](../../challenges/problems/36-word-search.ipynb), [Word Break](../../challenges/problems/53-word-break.ipynb).


## Common mistake

This bug is silent. Search returns true whenever the path exists, ignoring the end mark. After inserting only `"hi"`, `has_word("h")` prints `True`. The answer is `False`.


In [3]:
class BookWrong:
    def __init__(self):
        self.children = {}

    def add(self, text):
        node = self
        for character in text:
            node = node.children.setdefault(character, BookWrong())

    def has_word(self, text):
        node = self
        for character in text:
            if character not in node.children:
                return False
            node = node.children[character]
        return True  # any surviving path counts as a word


book = BookWrong()
book.add("hi")
print(book.has_word("h"))

True


The fix stores a `word` flag and returns that flag at the end of the path. The worked example does that, so the same query prints `False`.


In [4]:
book = Book()
book.add("hi")
print(book.has_word("h"))

False


## Python idioms

`setdefault` creates a child in one step. A separate `starts_with` walk is the same loop without reading the end mark. Sorting child keys gives a stable listing of stored words.


In [5]:
def stored_words(book: Book) -> list[str]:
    found: list[str] = []

    def walk(node: Book, prefix: str) -> None:
        if node.word:
            found.append(prefix)
        for character in sorted(node.children):
            walk(node.children[character], prefix + character)

    walk(book, "")
    return found


book = Book()
book.add("hi")
book.add("hat")
print(stored_words(book))

['hat', 'hi']


## Recap

- Edges are characters; nodes can mark word ends.
- Search needs the mark, not only a path.
- Shared prefixes share nodes.

Next: [practice](01-prefix-practice.ipynb).
